# Módulo 3 · Anotaciones: llevar las etiquetas de TUSZ a MNE

**Objetivo:** convertir los ficheros `.csv_bi` y `.csv` de TUSZ en `mne.Annotations`,
verlas sobre la señal y usarlas para seleccionar datos. Es la base de todas las etiquetas
(ictal, preictal, tipo de crisis…) del TFM.

### Recordatorio: los ficheros de anotación de TUSZ
| Fichero | Contenido | Granularidad |
|---|---|---|
| `.csv_bi` | `seiz` / `bckg` | un evento para todos los canales (`TERM`) |
| `.csv` | tipo de crisis (`absz`, `fnsz`, `gnsz`…) | **por canal del montaje TCP** |

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from tusz_utils import read_tusz_raw, find_edf

mne.set_log_level("WARNING")
mne.viz.set_browser_backend("matplotlib")

EDF = find_edf("aaaaabqv_s001_t000")
raw = read_tusz_raw(EDF, annotate=False).pick("eeg")
print(EDF)

## 3.1 Leer los CSV con pandas
Las primeras líneas empiezan por `#` (cabecera): `comment="#"` las salta.

In [ ]:
bi = pd.read_csv(EDF.with_suffix(".csv_bi"), comment="#")
bi

In [ ]:
multi = pd.read_csv(EDF.with_suffix(".csv"), comment="#")
print(multi.shape)
multi.head(10)

En el `.csv` hay **una fila por canal y por crisis** (22 canales × 3 crisis = 66 filas),
y los inicios pueden diferir unas décimas entre canales.

In [ ]:
multi.groupby("label").agg(n=("channel", "size"), canales=("channel", "nunique"),
                           ini_min=("start_time", "min"), fin_max=("stop_time", "max"))

## 3.2 `mne.Annotations`: onset, duration, description

Una anotación es solo una terna **(inicio en s, duración en s, texto)**.
Construimos las del `.csv_bi`:

In [ ]:
ann_bi = mne.Annotations(onset=bi.start_time.values,
                         duration=(bi.stop_time - bi.start_time).values,
                         description=bi.label.values)
ann_bi

> **`orig_time`**: si no lo indicas, los `onset` se interpretan como segundos desde el
> **inicio del fichero**, que es justo lo que dan los CSV de TUSZ.

In [ ]:
raw.set_annotations(ann_bi)
raw.annotations.to_data_frame(time_format=None)

Ahora `raw.plot()` sombrea las crisis automáticamente:

In [ ]:
fig = raw.plot(start=170, duration=140, n_channels=22, scalings=dict(eeg=200e-6),
               show_scrollbars=False, title="Las 3 crisis (seiz) del .csv_bi")

Dos detalles que se ven en esta figura y que importan:
- **A1 y A2 salen casi planos**: en el montaje *linked ears* (LE) las orejas *son* la referencia,
  así que su señal referida a sí mismas es ~0. En el módulo 4 lo arreglamos con el montaje bipolar.
- **OZ deriva mucho** (ondas lentas enormes fuera de las crisis): huele a **canal malo**.
  Lo trataremos en el módulo 6. Por suerte OZ no entra en el montaje TCP.

## 3.3 Anotaciones con el **tipo** de crisis (`.csv`)

Para tener una anotación por crisis con su tipo hay que **fusionar** las filas de los
distintos canales: agrupamos por etiqueta y unimos intervalos que se solapan.

In [ ]:
def merge_intervals(df, gap=0.0):
    out = []
    for label, g in df.sort_values("start_time").groupby("label"):
        s0 = e0 = None
        for s, e in zip(g.start_time, g.stop_time):
            if s0 is None:
                s0, e0 = s, e
            elif s <= e0 + gap:          # se solapa con el intervalo actual
                e0 = max(e0, e)
            else:                        # empieza uno nuevo
                out.append((s0, e0, label)); s0, e0 = s, e
        out.append((s0, e0, label))
    return pd.DataFrame(out, columns=["start_time", "stop_time", "label"]).sort_values("start_time")

sz = merge_intervals(multi[multi.label != "bckg"])
sz

In [ ]:
ann = mne.Annotations(sz.start_time, sz.stop_time - sz.start_time, sz.label)
raw.set_annotations(ann)
raw.annotations

## 3.4 Operar con anotaciones

`Annotations` se comporta como una lista: se puede iterar, indexar, sumar y filtrar.

In [ ]:
for a in raw.annotations:
    print(f"{a['description']}: {a['onset']:.1f} s → {a['onset'] + a['duration']:.1f} s  ({a['duration']:.1f} s)")

In [ ]:
print("Duraciones:", raw.annotations.duration)
print("Tiempo total en crisis:", raw.annotations.duration.sum(), "s de", raw.times[-1], "s")

In [ ]:
# Añadir una anotación a mano (por ejemplo, un tramo con artefacto)
raw.annotations.append(onset=50, duration=5, description="BAD_artefacto")
raw.annotations

> ⭐ **Convención importante:** las anotaciones cuya descripción empieza por **`BAD`**
> (`BAD_`, `bad_`…) se tratan como **tramos malos**. MNE los excluye automáticamente al crear
> epochs (`reject_by_annotation=True`) y en muchos cálculos (`compute_psd`, ICA…).

In [ ]:
# Borrar la anotación que acabamos de añadir
idx = [i for i, d in enumerate(raw.annotations.description) if d.startswith("BAD")]
raw.annotations.delete(idx)
raw.annotations

## 3.5 `crop` respeta las anotaciones

In [ ]:
tramo = raw.copy().crop(230, 300)
print(tramo.annotations)
tramo.annotations.to_data_frame(time_format=None)   # los onset siguen en tiempo del fichero original

## 3.6 De anotaciones a **eventos**

Muchas funciones de MNE (sobre todo `mne.Epochs`) trabajan con **eventos**: un array
`(n, 3)` con `[muestra, 0, id]`. `events_from_annotations` hace la conversión.

In [ ]:
events, event_id = mne.events_from_annotations(raw)
print(event_id)
print(events)
print("Muestra → segundos:", events[:, 0] / raw.info["sfreq"])

Truco muy útil: **`chunk_duration`** trocea cada anotación en eventos regulares. Así
obtienes, por ejemplo, un evento cada 1 s dentro de cada crisis:

In [ ]:
ev_chunks, _ = mne.events_from_annotations(raw, chunk_duration=1.0)
print(len(ev_chunks), "eventos de 1 s dentro de crisis")

## 3.7 Una máscara por muestra

Para segmentación (una etiqueta por instante) conviene un vector 0/1 del tamaño de la señal:

In [ ]:
def seizure_mask(raw):
    mask = np.zeros(raw.n_times, dtype=bool)
    sf = raw.info["sfreq"]
    for a in raw.annotations:
        if a["description"].startswith("BAD"):
            continue
        i0 = int((a["onset"] - raw.first_time) * sf)
        i1 = int((a["onset"] - raw.first_time + a["duration"]) * sf)
        mask[max(i0, 0):min(i1, raw.n_times)] = True
    return mask

m = seizure_mask(raw)
print(f"{m.mean() * 100:.2f}% de las muestras están en crisis")

plt.figure(figsize=(12, 1.6))
plt.fill_between(raw.times, m, step="mid", color="tab:red")
plt.yticks([]); plt.xlabel("Tiempo (s)"); plt.title("Máscara de crisis del fichero completo")
plt.tight_layout(); plt.show()

> Fíjate en el **desbalanceo**: ~1.5 % de crisis frente a ~98.5 % de fondo. Es el gran reto
> de la detección/predicción en TUSZ y lo trataremos en la fase 2 del TFM.

Todo lo de este módulo está empaquetado en `tusz_utils.tusz_annotations()` y
`read_tusz_raw(..., annotate=True)`:

In [ ]:
from tusz_utils import tusz_annotations
print(tusz_annotations(EDF, kind="bi"))
print(tusz_annotations(EDF, kind="multi"))

---
## 🧪 Ejercicios

**E3.1** Usando el inventario `seizures.csv` (en `Notas Dani/tusz_inventory_out`), lista todos
los ficheros con crisis `absz` y cuenta cuántas crisis y cuántos segundos hay por paciente.

**E3.2** Para el fichero de ejemplo, calcula el **intervalo entre crisis** (fin de una → inicio de la
siguiente) y el tiempo libre de crisis antes de la primera.

**E3.3** Abre `aaaaadqe_s001_t001`, añádele las anotaciones con `tusz_annotations` y dibuja
su máscara de crisis. ¿Qué porcentaje del registro es crisis?

### Solución E3.1

In [ ]:
from tusz_utils import INVENTORY
seiz = pd.read_csv(INVENTORY / "seizures.csv")
absz = seiz[seiz.label == "absz"].assign(dur=lambda d: d.stop - d.start)
absz.groupby(["split", "patient"]).agg(ficheros=("file", "nunique"), crisis=("dur", "size"),
                                       seg_total=("dur", "sum"), dur_media=("dur", "mean")).round(1)

### Solución E3.2

In [ ]:
on = raw.annotations.onset
off = on + raw.annotations.duration
print("Lead-in antes de la 1ª crisis:", round(on[0], 1), "s")
print("Intervalos entre crisis:", np.round(on[1:] - off[:-1], 1), "s")

### Solución E3.3

In [ ]:
r3 = read_tusz_raw("aaaaadqe_s001_t001")   # annotate=True por defecto
m3 = seizure_mask(r3)
print(r3.annotations, f"→ {m3.mean() * 100:.1f}% en crisis")
plt.figure(figsize=(12, 1.6)); plt.fill_between(r3.times, m3, step="mid", color="tab:red")
plt.yticks([]); plt.xlabel("s"); plt.show()